# Agent 第 07 课：Reflection（自反思）与错误恢复

目前我们的 agent 回答完就走——好坏都不管。但人类专家解题时会**回头检查**："这个答案对吗？有没有遗漏？" 如果有问题，改。

这就是 **Reflection**（自反思）模式。代表论文：*Reflexion*（2023）。核心循环：

```
  Attempt  →  Critique  →  (If bad) Retry with critique as feedback
```

实现上最简版本就是三个 LLM 调用：
1. **Actor**：正常生成答案
2. **Critic**：读问题 + 答案，输出 `{ok: bool, reasons: [...]}`
3. 如果 `not ok`，把 critic 的意见塞回 actor，让它重做

循环几次直到通过，或达到 max_retries。

In [ ]:
import json, re
from openai import OpenAI
client = OpenAI(base_url='http://10.0.0.63:1234/v1', api_key='lm-studio')
model_ids = [m.id for m in client.models.list().data]
chat_model = next((m for m in ['qwen/qwen3.6-35b-a3b','qwen3.6-35b-a3b','qwen3.5-35b-a3b'] if m in model_ids), model_ids[0])
print('chat =', chat_model)

## 1. Actor：生成草稿答案

In [ ]:
def actor(question, feedback=None):
    sys = 'You answer the user\'s question. If feedback from a critic is provided, incorporate it and fix your previous answer.'
    msgs = [{'role':'system','content':sys}]
    if feedback:
        msgs.append({'role':'user','content':f'Question: {question}\n\nCritic feedback on your previous answer:\n{feedback}\n\nProduce an improved answer.'})
    else:
        msgs.append({'role':'user','content':question})
    r = client.chat.completions.create(model=chat_model, temperature=0, messages=msgs)
    return r.choices[0].message.content.strip()

## 2. Critic：判断答案好不好

一个关键设计决定：**让 critic 输出结构化 JSON**。`{ok: bool, reasons: [...]}`。不要让它输出散文——你的循环没法判断停不停。

In [ ]:
CRITIC_SYSTEM = """You are a strict reviewer. You are given a user question and an assistant's answer.
Identify concrete problems: factual errors, missing parts, contradictions, unsupported claims, unclear reasoning.

Output strict JSON only:
{"ok": true|false, "score": 0-10, "reasons": ["...", "..."]}

Set ok=true only if the answer is correct AND addresses the question AND is clear. Otherwise false.
If ok=true, reasons can be empty."""

def critic(question, answer):
    usr = f'Question: {question}\n\nAnswer: {answer}\n\nReturn JSON only.'
    r = client.chat.completions.create(model=chat_model, temperature=0,
        messages=[{'role':'system','content':CRITIC_SYSTEM},
                  {'role':'user','content':usr}])
    txt = r.choices[0].message.content
    m = re.search(r'\{.*\}', txt, re.S)
    if not m: return {'ok':False,'score':0,'reasons':['bad critic output']}
    try: return json.loads(m.group(0))
    except Exception: return {'ok':False,'score':0,'reasons':['bad json']}

## 3. Reflection Loop

In [ ]:
def reflect_and_answer(question, max_retries=2, verbose=True):
    feedback = None
    last_answer = None
    for attempt in range(max_retries + 1):
        ans = actor(question, feedback=feedback)
        c = critic(question, ans)
        if verbose:
            print(f'--- attempt {attempt+1} ---')
            print('ANSWER:', ans[:200], '...' if len(ans)>200 else '')
            print('CRITIC:', c)
            print()
        if c.get('ok'):
            return ans, c, attempt+1
        feedback = '\n'.join(f'- {r}' for r in c.get('reasons', [])) or 'Improve the answer.'
        last_answer = ans
    return last_answer, c, max_retries+1

## 4. 故意问一个容易答错 / 答不全的问题

In [ ]:
q = 'List the four largest countries in the world by land area, in order, with approximate areas in km².'
answer, crit, n = reflect_and_answer(q, max_retries=2)
print('=== DONE in', n, 'attempts ===')
print(answer)

## 5. 把 Reflection 嵌入 ReAct Agent

上面是对"单次答案"做反思。更常见的是**对一整个 agent 执行过程做反思**：agent 跑完一轮，检查它的答案+trace，不满意就让它整体重来，并把"上次失败原因"作为 hint 传给新一轮。

这就是 Reflexion 论文的核心——**episodic memory**：每次失败的经验写进 memory，下次尝试时能看到。

简化版伪代码：
```
reflections = []
for attempt in range(max_attempts):
    answer, trace = run_agent(question, extra_hints=reflections)
    verdict = critic(question, answer, trace)
    if verdict.ok: return answer
    reflections.append(verdict.reasons)
```

## 6. Reflection 的坑

Reflection 看起来香，但有三个**必须知道的陷阱**：

### 坑 1：无限循环
Critic 可能永远 `ok=false`。必须有 `max_retries`，通常 2-3 就够。超过 3 次后往往陷入语言来回调整，质量不再提升。

### 坑 2：Critic 比 Actor 弱，反而拖后腿
如果 critic 用的是比 actor 更弱的模型，它会"批评出不存在的问题"，逼 actor 把正确答案改错。**Critic 至少要和 actor 一样强**。

### 坑 3：自反思在某些任务上无效
- **事实问答**：Critic 不知道真实答案，没法验证。必须配合工具（搜索、DB 查询）才有用。
- **数学/代码**：Reflection 很有用，因为有客观对错。
- **开放写作**：有用，但可能陷入"越改越平庸"的困境。

**经验法则**：
- 有客观 ground truth → Reflection 有用
- 没有 ground truth，只是"好不好" → 多半是拍脑袋，限制到 1 次 retry 就够
- 关键场景：**代码生成 + 执行** —— 让 critic 真的去**跑代码看测试是否通过**。这才是 reflection 的杀手级应用。

## 7. 执行-验证型 Reflection（推荐）

当 critic 可以**真的去执行**（跑代码、查数据库、调 API），reflection 效果立刻从"有点用"变成"质的飞跃"。

示例：让 agent 写一个 Python 函数，critic **实际运行**它的测试用例。我们写个最简版——让 actor 写代码，executor 跑测试，失败就把错误反馈回去重写。

In [ ]:
import subprocess, tempfile, textwrap, os

def run_python_code(code, test_code):
    full = code + '\n\n' + test_code
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(full); path=f.name
    try:
        r = subprocess.run(['python', path], capture_output=True, text=True, timeout=10)
        return r.returncode == 0, (r.stdout + r.stderr).strip()
    finally:
        os.unlink(path)

def code_actor(task, feedback=None):
    msgs=[{'role':'system','content':'You write Python. Output ONLY the function code, no explanation, no markdown fences.'}]
    if feedback:
        msgs.append({'role':'user','content':f'Task: {task}\n\nPrevious attempt failed:\n{feedback}\n\nWrite corrected code.'})
    else:
        msgs.append({'role':'user','content':task})
    r = client.chat.completions.create(model=chat_model, temperature=0, messages=msgs)
    code = r.choices[0].message.content.strip()
    # 去掉可能的 ```python fences
    code = re.sub(r'^```\w*\n', '', code); code = re.sub(r'\n```$', '', code)
    return code

def coding_agent(task, test_code, max_retries=3):
    feedback = None
    for i in range(max_retries+1):
        code = code_actor(task, feedback)
        ok, output = run_python_code(code, test_code)
        print(f'--- attempt {i+1} --- {"OK" if ok else "FAIL"}')
        print(code); print('output:', output[:200]); print()
        if ok: return code
        feedback = f'Tests failed with:\n{output}'
    return None

task = 'Write a function `fib(n)` that returns the n-th Fibonacci number. fib(0)=0, fib(1)=1.'
test = 'assert fib(0)==0\nassert fib(1)==1\nassert fib(10)==55\nassert fib(20)==6765\nprint("all pass")'
final = coding_agent(task, test)
print('=== FINAL CODE ===\n', final)

## 8. 工程直觉

1. **Reflection 的威力 = critic 能多严格**。能执行的 critic > LLM judge critic > "请自我检查"的 single-model reflection。
2. **Actor + Critic 用同一个模型也可以，但注意 prompt 要切换角色**——同一个模型在"写"和"评"两种 system prompt 下产出可以差很多（论文里叫 self-refine）。
3. **Reflection 是最贵的模式之一**。每次 retry 都是完整 LLM 调用。生产里通常只对"高价值、可验证"的任务开，比如代码生成。
4. **结合 memory**：每次失败的反思写进长期记忆（第 05 课），遇到类似任务时先 recall，让 agent 少犯同样的错。Reflexion 论文的核心贡献就是这个。
5. **不要过度反思**。max_retries=2 对多数任务足够。"让模型再改一遍"的边际收益在第 3 次之后基本归零。

---

下一课：**第 08 课 Multi-Agent 协作**——一个 agent 搞定不了，就让多个 agent 分工。Manager/Worker、Researcher/Writer/Reviewer 这些常见模式。